# Chapter 11 — Context Management in Production Systems · Practice

**Goal:** measure what fills a GPT-6 request, choose where conversation state lives, compact and trim history without breaking it, keep transactional facts outside the history, and make a long-running conversation survive a crash.

- Theory: [README.md](README.md) (sections 11.1–11.12). This notebook holds the code.
- **Estimated time:** 3–4 hours
- **Estimated API cost:** under $0.02 with the default `gpt-6-luna` (a handful of small `responses.create`, `responses.parse`, `responses.compact` and `input_tokens.count` calls). Every section also has offline cells that cost nothing.
- **Prerequisites:** [Chapter 01](../01-responses-api-fundamentals/README.md) (conversation state, tokens), [Chapter 03](../03-agents-sdk-and-agents-api/README.md) (`Runner`, sessions, `RunState`), the repo venv from [00-prerequisites](../../00-prerequisites/README.md), and `OPENAI_API_KEY` in `.env` for the live cells.

Cells marked **(live)** call the OpenAI API. Everything else runs offline: token counts are **estimates** (about 4 characters per token), the Agents SDK runs against a scripted stand-in model, and compaction uses a fake client that returns a real `CompactedResponse` object. Scratch files (SQLite sessions, checkpoints, scratchpads) go to a temporary directory, never into the repo.

In [ ]:
import asyncio
import copy
import hashlib
import itertools
import json
import math
import os
import re
import tempfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from openai.types.responses import CompactedResponse, Response
from pydantic import BaseModel, Field

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
WORKDIR = Path(tempfile.mkdtemp(prefix="oa_ch11_"))  # scratch space outside the repo
print("Model:", MODEL)
print("Scratch dir:", WORKDIR)

In [ ]:
# Live cells need a key. Offline cells keep working if you skip this one.
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
client = OpenAI()

### Shared helpers

Prices come from the GPT-6 model pages at the time of writing (USD per 1M tokens); check them before you quote a number to a client. `est_tokens` is a rough local estimate; the exact count comes from `client.responses.input_tokens.count(...)` (11.1).

In [ ]:
PRICES = {  # USD per 1M tokens: input, cached input (read), cache write, output
    "gpt-6-luna":  {"input": 0.10, "cached": 0.01, "write": 0.125, "output": 0.50},
    "gpt-6.1-sol": {"input": 2.00, "cached": 0.10, "write": 2.50,  "output": 10.00},
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "write": 12.50, "output": 50.00},
}
LONG_CONTEXT_THRESHOLD = 272_000  # above this, the WHOLE request is billed at 2x input/cache and 1.5x output


def est_tokens(obj: Any) -> int:
    """Rough estimate (~4 chars per token). Use responses.input_tokens.count for the exact number."""
    text = obj if isinstance(obj, str) else json.dumps(obj, default=str)
    return max(1, math.ceil(len(text) / 4))


def request_cost(model: str, input_tokens: int, output_tokens: int,
                 cached_tokens: int = 0, cache_write_tokens: int = 0) -> float:
    """USD for one request. Assumes cached and cache-write tokens are part of input_tokens."""
    p = PRICES[model]
    in_mult, out_mult = (2.0, 1.5) if input_tokens > LONG_CONTEXT_THRESHOLD else (1.0, 1.0)
    uncached = input_tokens - cached_tokens - cache_write_tokens
    usd_in = (uncached * p["input"] + cached_tokens * p["cached"] + cache_write_tokens * p["write"]) * in_mult
    return (usd_in + output_tokens * p["output"] * out_mult) / 1e6


def msg(role: str, text: str) -> dict:
    return {"type": "message", "role": role, "content": text}


def mock_response(output: list[dict], *, input_tokens: int = 1_000, cached: int = 0, writes: int = 0,
                  output_tokens: int = 50, reasoning: int = 0) -> Response:
    """A typed Response object around mock output items (no API call)."""
    return Response.model_validate({
        "id": f"resp_mock_{next(_SEQ)}", "object": "response", "created_at": 0, "model": MODEL,
        "output": output, "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
        "usage": {"input_tokens": input_tokens, "output_tokens": output_tokens,
                  "total_tokens": input_tokens + output_tokens,
                  "input_tokens_details": {"cached_tokens": cached, "cache_write_tokens": writes},
                  "output_tokens_details": {"reasoning_tokens": reasoning}},
    })


def assistant_item(text: str) -> dict:
    return {"type": "message", "id": f"msg_{next(_SEQ)}", "role": "assistant", "status": "completed",
            "content": [{"type": "output_text", "text": text, "annotations": []}]}


_SEQ = itertools.count(1)  # globally unique mock IDs (the Agents SDK de-duplicates items by ID)
print("helpers ready; a 10K-in / 1K-out request on Luna costs", f"${request_cost('gpt-6-luna', 10_000, 1_000):.5f}")

## 11.1 What fills the window, and what it costs

The window holds instructions, tool definitions, every input item, replayed reasoning, and this turn's reasoning plus output. Each turn re-sends (and re-bills) the whole history, so the total bill of a chat grows roughly with the **square** of the number of turns, and above 272K input tokens the whole request costs more. Measure before you send, measure after, and turn the numbers into a budget policy. [Theory](README.md#111-what-fills-the-window-and-what-it-costs)

In [ ]:
RULES = (
    "You are the support assistant for Acme Outdoor. Answer only from tool results and the case facts. "
    "Never promise refunds above 200 USD without a human. Be concise."
)
TOOLS = [
    {"type": "function", "name": "lookup_order", "description": "Look up an order by ID.",
     "parameters": {"type": "object", "properties": {"order_id": {"type": "string"}},
                    "required": ["order_id"], "additionalProperties": False}, "strict": True},
    {"type": "function", "name": "search_kb", "description": "Search the help-center articles.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}},
                    "required": ["query"], "additionalProperties": False}, "strict": True},
]


def full_order_record(order_id: str) -> dict:
    """What a real order API returns: 35 fields, most of them irrelevant to support."""
    rec = {"order_id": order_id, "status": "delivered", "total": "74.95 USD", "currency": "USD",
           "items": [{"sku": "TENT-2P", "name": "Trail tent 2P", "qty": 1}], "refund_window_ends": "2026-10-12",
           "customer_id": "CUST-881", "placed_at": "2026-09-12T10:04:00Z"}
    rec.update({f"warehouse_field_{i:02d}": f"value-{i}-" + "x" * 30 for i in range(27)})
    return rec


HISTORY: list[dict] = []
for turn in range(1, 6):
    HISTORY += [
        msg("user", f"Turn {turn}: question about order A-1001 and its return window."),
        {"type": "function_call", "call_id": f"call_t{turn}", "name": "lookup_order",
         "arguments": json.dumps({"order_id": "A-1001"})},
        {"type": "function_call_output", "call_id": f"call_t{turn}", "output": json.dumps(full_order_record("A-1001"))},
        msg("assistant", "Your order A-1001 (74.95 USD) was delivered and is eligible for return."),
    ]

parts = {
    "instructions": est_tokens(RULES),
    "tool definitions": est_tokens(TOOLS),
    "user + assistant messages": est_tokens([i for i in HISTORY if i.get("type") == "message"]),
    "function calls": est_tokens([i for i in HISTORY if i.get("type") == "function_call"]),
    "function call outputs": est_tokens([i for i in HISTORY if i.get("type") == "function_call_output"]),
}
total = sum(parts.values())
for name, n in sorted(parts.items(), key=lambda kv: -kv[1]):
    print(f"{name:<26} {n:>6} est. tokens  {n / total:6.1%}  {'#' * round(40 * n / total)}")
print(f"{'total':<26} {total:>6}")

> **Observe:** after only five turns, tool outputs are most of the context. The 35-field order record re-sent every turn is the tool-output growth described in README 11.1 and 11.7. Section 11.7 fixes it at the source.

The quadratic curve: if each turn adds `per_turn` tokens, turn *n* sends about `per_turn × n` input tokens.

In [ ]:
def chat_input_tokens(turns: int, per_turn: int) -> int:
    """Total input tokens billed over a chat in which every turn re-sends the whole history."""
    return sum(per_turn * n for n in range(1, turns + 1))


for per_turn in (3_000, 1_000):
    total_in = chat_input_tokens(60, per_turn)
    # each request is far below 272K here, so the base input price applies to every token
    costs = ", ".join(f"{m}: ${total_in * PRICES[m]['input'] / 1e6:,.2f}" for m in ("gpt-6-luna", "gpt-6-astra"))
    print(f"60 turns x {per_turn:>5} tokens/turn -> {total_in / 1e6:.2f}M input tokens ({costs}, before caching)")

print("\nCumulative input tokens (3,000 tokens per turn):")
for turns in range(10, 61, 10):
    n = chat_input_tokens(turns, 3_000)
    print(f"  {turns:>2} turns {n / 1e6:5.2f}M {'#' * round(n / 150_000)}")

> **Observe:** 60 turns at 3,000 tokens per turn bill 5.49M input tokens (about $0.55 on Luna, about $55 on Astra). Cutting each turn's growth to 1,000 tokens cuts the total by two thirds. Caching lowers the **price** of the repeated prefix (11.12); trimming and compaction lower the **size**.

Now the 272K threshold: one token across it changes the price of the whole request.

In [ ]:
for model in ("gpt-6-luna", "gpt-6-astra"):
    below = request_cost(model, 271_999, 2_000)
    above = request_cost(model, 272_001, 2_000)
    print(f"{model:<12} 271,999 in -> ${below:.4f}   272,001 in -> ${above:.4f}   (x{above / below:.2f})")

A **context budget** turns the numbers into a policy (same constants as the README). The soft limit leaves room to compact while compaction is still possible.

In [ ]:
WINDOW_CEILING = 272_000          # pricing threshold, not the 1.05M technical window
OUTPUT_RESERVE = 25_000           # room for reasoning + visible output
SOFT_LIMIT = int(0.6 * (WINDOW_CEILING - OUTPUT_RESERVE))
HARD_LIMIT = WINDOW_CEILING - OUTPUT_RESERVE


def next_action(input_tokens: int) -> str:
    if input_tokens >= HARD_LIMIT:
        return "compact_now"      # or delegate / start fresh from case facts; never send as-is
    if input_tokens >= SOFT_LIMIT:
        return "compact"
    return "send"


print(f"SOFT_LIMIT={SOFT_LIMIT:,}  HARD_LIMIT={HARD_LIMIT:,}")
for n in (40_000, SOFT_LIMIT - 1, SOFT_LIMIT, 200_000, HARD_LIMIT, 400_000):
    print(f"  {n:>8,} input tokens -> {next_action(n)}")

**Measure after you send.** `response.usage` carries everything you need to log per turn. Reasoning tokens are **inside** `output_tokens`, and cached and cache-write tokens are inside `input_tokens`.

In [ ]:
def usage_row(resp: Response) -> dict:
    u = resp.usage
    return {
        "input": u.input_tokens,
        "cached": u.input_tokens_details.cached_tokens,
        "cache_write": u.input_tokens_details.cache_write_tokens,
        "output": u.output_tokens,
        "reasoning (in output)": u.output_tokens_details.reasoning_tokens,
        "cache_hit_rate": round(u.input_tokens_details.cached_tokens / max(1, u.input_tokens), 2),
        "next_action": next_action(u.input_tokens),
    }


turns = [mock_response([assistant_item("ok")], input_tokens=n, cached=c, writes=w, output_tokens=400, reasoning=250)
         for n, c, w in [(12_000, 0, 11_000), (15_500, 11_000, 3_500), (160_000, 150_000, 0)]]
for i, r in enumerate(turns, 1):
    print(f"turn {i}:", usage_row(r))

**(live)** `client.responses.input_tokens.count(...)` accepts the same payload as `responses.create` (also `previous_response_id` or `conversation`) and returns the exact count, including role and message-boundary tokens a local estimate cannot see. Counting is free of model output; compare it with the estimate.

In [ ]:
# (live) exact count for the 5-turn request above
count = client.responses.input_tokens.count(
    model=MODEL, instructions=RULES, tools=TOOLS,
    input=HISTORY + [msg("user", "Can I still return it?")],
)
estimate = est_tokens(RULES) + est_tokens(TOOLS) + est_tokens(HISTORY)
print(f"exact: {count.input_tokens:,}   local estimate: {estimate:,}   ratio: {estimate / count.input_tokens:.2f}")

> **Try it:** change `OUTPUT_RESERVE` to 0 and see how close `HARD_LIMIT` gets to the pricing threshold. Then set the soft limit from your own evals: README 11.1 notes that quality usually degrades long before the window is full.

**A cheap lever for long replays: `reasoning.context`.** With `"current_turn"`, earlier reasoning items may stay in the payload, but the service does not render them into the next sample (the reasoning guide: it "can reduce the rendered context for long-running workflows"). `"all_turns"` renders them; the reasoning guide lists it for the GPT-5.6 models (their default) and `gpt-6.1-sol`. Omit the field (`"auto"`) to use the selected model's default. The SDK's `Reasoning` type accepts the field; the response's `reasoning.context` reports the mode actually used.

In [ ]:
from openai.types.shared_params import Reasoning

assert "context" in Reasoning.__annotations__, "this openai SDK predates reasoning.context"


def replay_request(history: list[dict], user_message: str, *, keep_old_reasoning: bool) -> dict:
    """Stateless replay of the full history; only the rendering of older reasoning differs."""
    return {
        "model": MODEL, "store": False, "instructions": RULES,
        "input": [*history, msg("user", user_message)],
        "reasoning": {"context": "all_turns" if keep_old_reasoning else "current_turn"},
    }


for keep in (True, False):
    r = replay_request(HISTORY, "Summarize where we are.", keep_old_reasoning=keep)
    print(f"reasoning.context={r['reasoning']['context']:<12} input items sent: {len(r['input'])} (same payload, "
          f"{'older reasoning rendered' if keep else 'older reasoning NOT rendered'})")

## 11.2 Conversation state for long-running work: who can edit the history?

For context management the deciding question is whether you can **curate** what the model sees next turn. Stateless requests and Agents SDK sessions are fully editable; a `previous_response_id` chain is immutable; a Conversation allows deleting and adding items. Pick one strategy per conversation, because mixing them duplicates context. [Theory](README.md#112-conversation-state-for-long-running-work-who-can-edit-the-history)

A fifth, managed option sits outside this table: on the **Agents API** (beta, the managed Codex harness) "OpenAI manages sessions, orchestration, context compaction, and recovery". Less code, but even less control over what is kept, so the case-facts block and your own checkpoints still apply.

In [ ]:
NEW = msg("user", "Can I still return it?")


def request_for(strategy: str) -> dict:
    """The turn-6 request each strategy sends (offline payloads, nothing is sent)."""
    base = {"model": MODEL, "instructions": RULES, "tools": TOOLS}
    if strategy == "stateless":
        return {**base, "store": False, "input": [*HISTORY, NEW]}           # you replay everything
    if strategy == "chained":
        return {**base, "previous_response_id": "resp_turn5", "input": [NEW]}  # server holds the chain
    if strategy == "durable":
        return {**base, "conversation": "conv_case_4711", "input": [NEW]}      # Conversations API
    raise ValueError(strategy)


EDITABLE = {
    "stateless": "yes: you build input every turn",
    "chained": "no: the chain is immutable; start a new chain seeded with curated context",
    "durable": "partly: conversations.items.delete / items.create",
    "agents_session": "yes: get_items / add_items / pop_item / clear_session + input callbacks",
}
for s in ("stateless", "chained", "durable"):
    r = request_for(s)
    print(f"{s:<10} sends ~{est_tokens(r):>6,} est. tokens in the request body | editable: {EDITABLE[s]}")
print(f"{'session':<10} Agents SDK loads history from YOUR store | editable: {EDITABLE['agents_session']}")

> **Observe:** the chained and durable request bodies are small, but the model still **reads and bills** the whole stored history. The body size is not the context size; `input_tokens.count(previous_response_id=...)` measures the real thing.

A small linter catches the strategy mistakes the README lists.

In [ ]:
def lint_state(request: dict, *, uses_agents_session: bool = False) -> list[str]:
    problems = []
    if request.get("previous_response_id") and request.get("conversation"):
        problems.append("previous_response_id and conversation cannot be combined in one request")
    if uses_agents_session and (request.get("previous_response_id") or request.get("conversation")):
        problems.append("an Agents SDK session cannot be combined with conversation_id / previous_response_id")
    if request.get("previous_response_id") and not request.get("instructions"):
        problems.append("instructions are per request and are NOT carried by previous_response_id: send them again")
    if request.get("previous_response_id") and len(request.get("input", [])) > 1:
        problems.append("chained requests should send only the new turn; replaying history duplicates context")
    if request.get("store") is False and request.get("previous_response_id"):
        problems.append("store=False responses are not stored, so a later turn cannot chain to them")
    return problems


cases = {
    "stateless (good)": request_for("stateless"),
    "chained (good)": request_for("chained"),
    "chained, forgot instructions": {k: v for k, v in request_for("chained").items() if k != "instructions"},
    "chained + replayed history": {**request_for("chained"), "input": [*HISTORY, NEW]},
    "chain + conversation": {**request_for("chained"), "conversation": "conv_1"},
}
for name, req in cases.items():
    print(f"{name:<30} {lint_state(req) or 'OK'}")

An Agents SDK session in **your** storage is editable item by item. A file path makes it survive a restart; `SQLiteSession("id")` without a path is in-memory and per process.

In [ ]:
from agents import SQLiteSession

DB = WORKDIR / "support.db"
session = SQLiteSession("case_4711", DB)
await session.add_items([msg("user", "My card number is 4111 1111 1111 1111, order A-1001."),
                         msg("assistant", "Thanks, I found order A-1001.")])
print("stored:", [i["role"] for i in await session.get_items()])

# Curate: remove the message with card data, store a redacted version instead.
assistant = await session.pop_item()
await session.pop_item()
await session.add_items([msg("user", "[card number redacted], order A-1001."), assistant])
print("after redaction:", [i["content"][:40] for i in await session.get_items()])

reopened = SQLiteSession("case_4711", DB)   # a new process would do exactly this
print("reopened from disk:", len(await reopened.get_items()), "items")

> **Try it:** ask yourself how you would do the same redaction with a `previous_response_id` chain. (You cannot: start a new chain or a stateless request seeded with the curated context.)

## 11.3 Truncation: the blunt instrument

`truncation="disabled"` (the default) makes an oversized request fail with HTTP 400; `truncation="auto"` silently drops items **from the beginning** of the conversation until it fits. The beginning is where customers give their account and order numbers. [Theory](README.md#113-truncation-the-blunt-instrument)

The simulation below uses a toy window of 1,000 estimated tokens so that the effect is visible.

In [ ]:
class ContextOverflow(Exception):
    """Stands in for the HTTP 400 you get with truncation='disabled'."""


def simulate_send(items: list[dict], window: int, truncation: str = "disabled") -> list[dict]:
    items = list(items)
    if est_tokens(items) <= window:
        return items
    if truncation == "disabled":
        raise ContextOverflow(f"input ~{est_tokens(items)} tokens exceeds the {window}-token window")
    while items and est_tokens(items) > window:   # 'auto': drop from the beginning
        items.pop(0)
    return items


chat = [msg("user", "Hi, my order number is A-1001 and the tent arrived torn.")]
for i in range(1, 25):
    chat += [msg("assistant", f"Reply {i}: " + "Let me check that for you. " * 6), msg("user", f"Follow-up {i}.")]

for mode in ("disabled", "auto"):
    try:
        sent = simulate_send(chat, window=1_000, truncation=mode)
        still_there = any("A-1001" in str(i["content"]) for i in sent)
        print(f"{mode:<8} sent {len(sent)}/{len(chat)} items; order number still in context: {still_there}")
    except ContextOverflow as e:
        print(f"{mode:<8} ContextOverflow: {e}")

> **Observe:** `disabled` fails loudly, which is what you want: a 400 for context overflow is a bug in your budget policy, not a retryable error. `auto` keeps the service up and silently forgets the order number. The Agents SDK exposes the same setting as `ModelSettings(truncation=...)`.

In [ ]:
from agents import ModelSettings

print(ModelSettings(truncation="disabled"))

## 11.4 Compaction: server-side and standalone

**Server-side compaction** runs inside `responses.create` when the rendered context crosses `compact_threshold` and emits an **encrypted, opaque** compaction item. **Standalone compaction** (`client.responses.compact`) is an explicit, billed call that returns the canonical next window. The pruning rules differ per mode, and you can never read what the compaction item kept. [Theory](README.md#114-compaction-server-side-and-standalone)

In [ ]:
def server_side_request(history: list[dict], user_message: str, threshold: int = 150_000) -> dict:
    """Stateless request with server-side compaction (ZDR-friendly with store=False)."""
    return {
        "model": MODEL,
        "instructions": RULES,
        "input": [*history, msg("user", user_message)],
        "store": False,
        "context_management": [{"type": "compaction", "compact_threshold": threshold}],
    }


req = server_side_request(HISTORY, "Can I still return it?")
print(json.dumps({k: v for k, v in req.items() if k not in ("input", "instructions")}, indent=2))
print("input items:", len(req["input"]))

Suppose the threshold was crossed. The response contains a compaction item followed by the answer. In stateless mode you append the output as usual, and **may** drop everything before the most recent compaction item.

In [ ]:
OPAQUE = "gAAAAB" + hashlib.sha256(b"whatever the server kept").hexdigest() * 4   # stands in for encrypted_content
resp = mock_response([{"type": "compaction", "id": "cmp_1", "encrypted_content": OPAQUE},
                      assistant_item("Yes, A-1001 is eligible for return until 2026-10-12.")],
                     input_tokens=152_000, output_tokens=60)
print("output item types:", [o.type for o in resp.output])

conversation = [*HISTORY, msg("user", "Can I still return it?")]
conversation += [o.model_dump(exclude_none=True) for o in resp.output]   # append output items as usual


def drop_before_latest_compaction(items: list[dict]) -> list[dict]:
    """Stateless chaining only: the latest compaction item carries the context needed so far."""
    last = max((i for i, it in enumerate(items) if it.get("type") == "compaction"), default=None)
    return items if last is None else items[last:]


pruned = drop_before_latest_compaction(conversation)
print(f"before pruning: {len(conversation)} items (~{est_tokens(conversation):,} est. tokens)")
print(f"after pruning:  {len(pruned)} items (~{est_tokens(pruned):,} est. tokens) -> {[i['type'] for i in pruned]}")

Two rules to remember: with `previous_response_id` chaining you pass only the new user message and **do not prune**; and the output of standalone compaction is never pruned. And whatever mode you use, the compaction item is opaque:

In [ ]:
item = pruned[0]
print("encrypted_content preview:", item["encrypted_content"][:48], "...")
print("Can we find the refund amount '74.95' in it?", "74.95" in item["encrypted_content"])

> **Observe:** you cannot audit, diff or migrate what compaction kept. That is why the case-facts block (11.6) and your own checkpoints (11.11) are mandatory on OpenAI, not optional.

**Standalone compaction** gives you control over *when*. The fake client below returns a real `CompactedResponse` object, so the code that consumes it is the same code you would run live.

In [ ]:
def fake_compacted(n_input_items: int) -> CompactedResponse:
    return CompactedResponse.model_validate({
        "id": f"cmp_resp_{next(_SEQ)}", "created_at": 0, "object": "response.compaction",
        "output": [{"type": "compaction", "id": f"cmp_{next(_SEQ)}", "encrypted_content": OPAQUE}],
        "usage": {"input_tokens": 400 * n_input_items, "output_tokens": 900, "total_tokens": 400 * n_input_items + 900,
                  "input_tokens_details": {"cached_tokens": 0, "cache_write_tokens": 0},
                  "output_tokens_details": {"reasoning_tokens": 600}},
    })


compacted = fake_compacted(len(HISTORY))       # live: client.responses.compact(model=MODEL, input=HISTORY)
next_input = [
    *[o.model_dump(exclude_none=True) for o in compacted.output],   # canonical next window: pass it on AS-IS
    msg("user", "Can I still return it?"),
]
print("compaction pass usage:", compacted.usage.input_tokens, "in /", compacted.usage.output_tokens, "out (billed)")
print("next input:", [i.get("type") for i in next_input])

The real returned window "generally contains more than just the compaction item" (it can include retained items, such as user messages). Either way, pass it on unchanged.

Which state strategies can the standalone endpoint compact? Ask the installed SDK (offline): `compact` takes `input` or `previous_response_id`, but no `conversation`.

In [ ]:
import inspect

from openai.resources.responses import Responses

_PLUMBING = {"self", "extra_headers", "extra_query", "extra_body", "timeout"}
compact_params = sorted(set(inspect.signature(Responses.compact).parameters) - _PLUMBING)
create_params = set(inspect.signature(Responses.create).parameters)
print("responses.compact accepts:", compact_params)
print("'conversation' on create:", "conversation" in create_params, "| on compact:", "conversation" in compact_params)
print("-> a Conversations-API thread has no documented compaction path: test before you combine them")

One more constraint from the reasoning guide: `configuration_update` items (used to change reasoning effort mid-conversation, 11.12) must not be combined with **automatic** compaction or truncation, and `/responses/compact` rejects histories that contain them. You can still compact explicitly by sending a `compaction_trigger` item as the **final** input item of a `/responses` request, then add a fresh `configuration_update` before the next user message.

In [ ]:
def lint_compaction(request: dict) -> list[str]:
    items = request.get("input", [])
    has_config_update = any(isinstance(i, dict) and i.get("type") == "configuration_update" for i in items)
    automatic = bool(request.get("context_management")) or request.get("truncation") == "auto"
    problems = []
    if has_config_update and automatic:
        problems.append("configuration_update cannot be combined with automatic compaction/truncation; "
                        "use an explicit compaction_trigger item instead")
    triggers = [n for n, i in enumerate(items) if isinstance(i, dict) and i.get("type") == "compaction_trigger"]
    if triggers and triggers[-1] != len(items) - 1:
        problems.append("compaction_trigger must be the final input item")
    if request.get("previous_response_id") and len(items) > 1:
        problems.append("previous_response_id chaining: send only the new turn and do not prune manually")
    return problems


effort_up = {"type": "configuration_update", "reasoning": {"effort": "high"}}
checks = {
    "automatic compaction + effort update": {**req, "input": [*HISTORY, effort_up, msg("user", "Deep-dive please.")]},
    "explicit trigger, last item": {"model": MODEL, "store": False, "input": [*HISTORY, {"type": "compaction_trigger"}]},
    "trigger not last": {"model": MODEL, "input": [{"type": "compaction_trigger"}, msg("user", "hi")]},
    "plain server-side compaction": req,
}
for name, r in checks.items():
    print(f"{name:<38} {lint_compaction(r) or 'OK'}")

**(live)** Standalone compaction of a short history, then one stateless turn on the compacted window. Expect a few hundred tokens of usage for each call.

In [ ]:
# (live) compact a short history, then continue statelessly on the returned window
short_history = [msg("user", "My order A-1001 (74.95 USD) arrived with a torn tent."),
                 msg("assistant", "Sorry to hear that. I can arrange a replacement or a refund."),
                 msg("user", "A refund please, to the original card.")]
compacted = client.responses.compact(model=MODEL, input=short_history)
print("compacted window:", [o.type for o in compacted.output], "| usage:", compacted.usage.input_tokens, "in")
follow_up = client.responses.create(
    model=MODEL, store=False,
    reasoning={"effort": "low"},      # a deliberate choice: Luna's default effort is medium
    max_output_tokens=1_000,          # room for reasoning + answer; a tight cap can end in status="incomplete"
    input=[*[o.model_dump(exclude_none=True) for o in compacted.output],
           msg("user", "Remind me: which order and amount are we refunding?")],
)
print(follow_up.status, "|", follow_up.output_text or follow_up.incomplete_details)

> **Try it:** run the live cell twice and compare the answers. The model usually recalls the order and the amount, but nothing guarantees it, and you cannot inspect why. Section 11.6 takes that risk away.

## 11.5 Long runs in the Agents SDK: limits, filters, trimmers and compacting sessions

The Agents SDK gives you four composable hooks: `SessionSettings(limit=N)` (load only the latest N items), `session_input_callback` (combine history and new input once per run), `call_model_input_filter` (edit input and instructions **before every model call**, including between tool calls) with the built-in `ToolOutputTrimmer`, and `OpenAIResponsesCompactionSession` (calls `responses.compact` after a turn when your hook says so). [Theory](README.md#115-long-runs-in-the-agents-sdk-limits-filters-trimmers-and-compacting-sessions)

To run agents without a key, the next cell defines a **scripted model**: a stand-in that implements the Agents SDK `Model` interface, replays a fixed plan of tool calls, and records exactly what it was sent. Swap it for `model=MODEL` to go live.

In [ ]:
from agents import Agent, ModelResponse, RunConfig, Runner, SessionSettings, Usage, function_tool
from agents.models.interface import Model
from openai.types.responses import ResponseFunctionToolCall, ResponseOutputMessage, ResponseOutputText


class ScriptedModel(Model):
    """Offline stand-in for an LLM: replays (tool_name, arguments) steps, then answers with `final`."""

    def __init__(self, plan: list[tuple[str, dict]] | None = None, final: str = "Done."):
        self.plan, self.final, self.calls = list(plan or []), final, []

    async def get_response(self, system_instructions, input, model_settings, tools, output_schema, handoffs,
                           tracing, *, previous_response_id=None, conversation_id=None, prompt=None):
        self.calls.append({"instructions": system_instructions, "input": copy.deepcopy(input)})
        n = next(_SEQ)
        if self.plan:
            name, args = self.plan.pop(0)
            item = ResponseFunctionToolCall(type="function_call", id=f"fc_{n}", call_id=f"call_{n}",
                                            name=name, arguments=json.dumps(args), status="completed")
        else:
            item = ResponseOutputMessage(type="message", id=f"msg_{n}", role="assistant", status="completed",
                                         content=[ResponseOutputText(type="output_text", text=self.final, annotations=[])])
        return ModelResponse(output=[item], usage=Usage(requests=1, input_tokens=est_tokens(input), output_tokens=20),
                             response_id=f"resp_{n}")

    def stream_response(self, *args, **kwargs):
        raise NotImplementedError("ScriptedModel supports Runner.run only")


def show_input(items: list, width: int = 70) -> None:
    for it in items:
        kind = it.get("type") or "message"
        role = it.get("role", "")
        body = it.get("output") if kind == "function_call_output" else it.get("content", it.get("arguments", ""))
        print(f"   {kind:<21} {role:<9} {str(body)[:width]!r}")


OFFLINE = RunConfig(tracing_disabled=True)


@function_tool
def search_kb(query: str) -> str:
    """Search the help-center articles. Returns long article text."""
    return f"Article for '{query}': " + "Returns are accepted within 30 days of delivery if unused. " * 40

print("tool ready:", search_kb.name, "(returns a ~2,400-character article per call)")

Three turns, each calling `search_kb`, with `ToolOutputTrimmer(recent_turns=1, ...)`. Look at what the model actually received on the last call.

In [ ]:
from agents.extensions import ToolOutputTrimmer
from agents.run_config import CallModelData, ModelInputData


def is_error_output(item: Any) -> bool:
    """A function_call_output whose JSON payload has an "error" key."""
    if not (isinstance(item, dict) and item.get("type") == "function_call_output"):
        return False
    try:
        data = json.loads(item.get("output") or "")
    except (TypeError, ValueError):
        return False
    return isinstance(data, dict) and "error" in data


def preserve_errors(trimmer):
    """Wrap a call_model_input_filter so error outputs always reach the model unchanged.
    ToolOutputTrimmer has no error exemption: keeping errors readable is application code."""
    def _filter(data: CallModelData) -> ModelInputData:
        original, trimmed = data.model_data.input, trimmer(data)
        assert len(trimmed.input) == len(original), "a trimmer must not add or remove items"
        items = [o if is_error_output(o) else t for o, t in zip(original, trimmed.input)]
        return ModelInputData(input=items, instructions=trimmed.instructions)
    return _filter


trim_config = RunConfig(
    tracing_disabled=True,
    session_settings=SessionSettings(limit=200),                   # never load more than 200 items
    call_model_input_filter=preserve_errors(ToolOutputTrimmer(recent_turns=1, max_output_chars=500, preview_chars=60,
                                                              trimmable_tools={"search_kb"})),
)
kb_session = SQLiteSession("kb_demo", WORKDIR / "kb.db")
for i, q in enumerate(["return window", "torn tent", "refund timing"]):
    model = ScriptedModel([("search_kb", {"query": q})], final=f"Answer {i + 1}.")
    agent = Agent(name="Support", instructions=RULES, model=model, tools=[search_kb])
    await Runner.run(agent, f"Question {i + 1}: {q}?", session=kb_session, run_config=trim_config)

print("What the model saw on the final call of turn 3:")
show_input(model.calls[-1]["input"])
stored = await kb_session.get_items()
print("\nStored in the session (untrimmed):", sum(len(str(i.get("output", ""))) for i in stored), "chars of tool output")

> **Observe:** the two older `search_kb` outputs reached the model as `[Trimmed: search_kb output — N chars → 60 char preview]`, the current turn's output is intact, every `function_call` still has its `function_call_output`, and the **stored** session is unchanged (the filter edits only what is sent).

`ToolOutputTrimmer` has no exemption for errors: an old `{"detail": ..., "error": ..., "retryable": ...}` output becomes a preview of its first characters, and the fields that explain why the agent stopped can disappear. That is why the cell above wraps it with `preserve_errors`:

In [ ]:
probe = [msg("user", "And order B-9?"),
         {"type": "function_call", "call_id": "call_e", "name": "lookup_order", "arguments": '{"order_id": "B-9"}'},
         {"type": "function_call_output", "call_id": "call_e",
          "output": json.dumps({"detail": "x" * 600, "error": "order B-9 not found", "retryable": False})},
         msg("assistant", "I could not find B-9."),
         msg("user", "OK, what about A-1001?")]
data = CallModelData(model_data=ModelInputData(input=probe, instructions=None), agent=Agent(name="probe"), context=None)
plain = ToolOutputTrimmer(recent_turns=1, max_output_chars=500, preview_chars=60)
for name, flt in (("ToolOutputTrimmer alone", plain), ("preserve_errors(ToolOutputTrimmer)", preserve_errors(plain))):
    out = flt(data).input[2]["output"]
    print(f"{name:<36} 'error' kept: {'order B-9 not found' in out!s:<5}  'retryable' kept: {'retryable' in out}")

One more limit: `ToolOutputTrimmer` never touches anything at or after the most recent `recent_turns` user messages, and `recent_turns` must be at least 1. Inside a **single** run (one user message, many tool calls) it therefore trims nothing, and that is deliberate. OpenAI's reasoning guide says that when you truncate or optimize context, all items **between the last user message and your latest function call output** must be passed to the next request untouched; reasoning items depend on them. So do not write a filter that rewrites outputs the model already received in this turn. Shrink them **before** they enter the context: return a bounded digest at the source (the full data stays in your storage behind a reference), delegate the exploration to an agent-as-tool (11.10), or let supported compaction (server-side `context_management`, 11.4) handle the growth.

In [ ]:
def raw_logs(service: str) -> str:
    log = f"[{service}] " + "2026-10-03T12:00:00Z INFO request ok\n" * 120
    return log + ("2026-10-03T12:00:03Z ERROR connection pool exhausted\n" if service == "db" else "")


LOG_STORE: dict[str, str] = {}   # full logs stay in YOUR storage, addressable by reference


@function_tool
def fetch_logs(service: str) -> str:
    """Fetch recent logs for a service (verbose)."""
    return raw_logs(service)


@function_tool
def fetch_log_digest(service: str) -> str:
    """Fetch recent logs for a service. Returns the line count, the ERROR lines and a reference to the full log."""
    full = raw_logs(service)
    ref = f"logs://{service}/2026-10-03T12:00"
    LOG_STORE[ref] = full
    lines = full.splitlines()
    return json.dumps({"service": service, "lines": len(lines),
                       "errors": [ln for ln in lines if " ERROR " in ln][:20], "full_log_ref": ref})


for name, tool, flt in (("raw tool + ToolOutputTrimmer", fetch_logs, ToolOutputTrimmer(recent_turns=1, max_output_chars=500)),
                        ("bounded digest at the source", fetch_log_digest, None)):
    model = ScriptedModel([(tool.name, {"service": s}) for s in ("api", "db", "queue", "auth")], final="Root cause: db.")
    agent = Agent(name="SRE", instructions="Investigate.", model=model, tools=[tool])
    await Runner.run(agent, "Why did checkout fail at 12:00?",
                     run_config=RunConfig(tracing_disabled=True, call_model_input_filter=flt))
    sizes = [len(it["output"]) for it in model.calls[-1]["input"] if it.get("type") == "function_call_output"]
    print(f"{name:<30} tool-output sizes on the final model call: {sizes}")

> **Observe:** with one user message and four tool calls, the built-in trimmer correctly left every output of the current turn at full size; the digest tool kept each output small from the start, and the ERROR line is still there. Use `ToolOutputTrimmer` (wrapped with `preserve_errors`) for older turns of multi-turn chats; inside one long run, shrink at the source, delegate, or compact.

`session_input_callback` runs once per `Runner.run`, before the loop. Use it to decide which stored history to combine with the new input.

In [ ]:
seen = {}


def keep_recent(history: list, new_input: list, max_items: int = 4) -> list:
    """Keep the most recent COMPLETE turns (each starts at a user message) that fit in max_items.
    Cutting at a turn boundary never splits a function_call from its output, even with parallel calls."""
    seen["history_items"] = len(history)
    starts = [i for i, it in enumerate(history) if it.get("role") == "user"]
    fitting = [i for i in starts if len(history) - i <= max_items]
    # if even the latest turn is longer than max_items, keep that whole turn: valid input beats a count
    start = fitting[0] if fitting else (starts[-1] if starts else len(history))
    recent = history[start:]
    calls = {it["call_id"] for it in recent if it.get("type") == "function_call"}
    orphans = [it["call_id"] for it in recent if it.get("type") == "function_call_output" and it["call_id"] not in calls]
    assert not orphans, f"orphaned tool outputs: {orphans}"
    return recent + new_input


# Why not simply history[-4:]? With parallel calls a fixed cut can keep an output without its call:
parallel = [msg("user", "Compare A-1 and B-2."),
            {"type": "function_call", "call_id": "call_a", "name": "lookup_order", "arguments": '{"order_id": "A-1"}'},
            {"type": "function_call", "call_id": "call_b", "name": "lookup_order", "arguments": '{"order_id": "B-2"}'},
            {"type": "function_call_output", "call_id": "call_a", "output": "{}"},
            {"type": "function_call_output", "call_id": "call_b", "output": "{}"},
            msg("assistant", "Both shipped.")]
naive = parallel[-4:]
print("naive last-4 keeps:", [i.get("call_id") or i["role"] for i in naive],
      "-> orphan outputs:", sorted({i["call_id"] for i in naive if i.get("type") == "function_call_output"}
                                   - {i["call_id"] for i in naive if i.get("type") == "function_call"}))
print("turn-based keeps:  ", [i.get("call_id") or i["role"] for i in keep_recent(parallel, [])])
seen.clear()

model = ScriptedModel(final="OK.")
agent = Agent(name="Support", instructions=RULES, model=model)
await Runner.run(agent, "Thanks!", session=kb_session,
                 run_config=RunConfig(tracing_disabled=True, session_settings=SessionSettings(limit=6),
                                      session_input_callback=keep_recent))
print("history loaded (limit=6):", seen["history_items"], "-> sent to the model:", len(model.calls[0]["input"]), "items")
show_input(model.calls[0]["input"], width=50)

A custom `call_model_input_filter` has the signature `(CallModelData) -> ModelInputData` (both from `agents.run_config`). It is the right place to inject a fresh case-facts block before **every** model call. Compose it with the trimmer:

In [ ]:
CURRENT_FACTS = {"text": '<case_facts version="3">\norder_id: A-1001\norder_total: 74.95 USD\n</case_facts>'}
trimmer = preserve_errors(ToolOutputTrimmer(recent_turns=1, max_output_chars=500, preview_chars=60))


def facts_and_trim(data: CallModelData) -> ModelInputData:
    trimmed = trimmer(data)                                   # 1) trim older tool outputs (errors kept)
    facts = {"role": "developer", "content": CURRENT_FACTS["text"]}
    return ModelInputData(input=[*trimmed.input, facts], instructions=trimmed.instructions)  # 2) facts at the end


model = ScriptedModel([("search_kb", {"query": "refund to card"})], final="Refund goes to the original card.")
agent = Agent(name="Support", instructions=RULES, model=model, tools=[search_kb])
await Runner.run(agent, "Where does the refund go?", session=kb_session,
                 run_config=RunConfig(tracing_disabled=True, call_model_input_filter=facts_and_trim))
for n, call in enumerate(model.calls, 1):
    last = call["input"][-1]
    print(f"model call {n}: {len(call['input'])} items, last item = {last.get('role')}: {str(last.get('content'))[:40]!r}")
print("facts stored in the session?", any("case_facts" in str(i) for i in await kb_session.get_items()))

> **Observe:** both model calls (before and after the tool call) got the facts block as the last item, and the facts never entered the stored history. The facts are ephemeral, the history stays append-only.

`OpenAIResponsesCompactionSession` wraps another session and calls `responses.compact` after a turn when `should_trigger_compaction` returns `True`. Its defaults (model `"gpt-4.1"`, trigger at 10 candidate items) are rarely what you want, so set both. Offline, a fake async client returns a real `CompactedResponse`.

In [ ]:
from agents.memory import OpenAIConversationsSession, OpenAIResponsesCompactionSession


class FakeAsyncResponses:
    def __init__(self):
        self.calls: list[dict] = []

    async def compact(self, **kwargs) -> CompactedResponse:
        self.calls.append(kwargs)
        return fake_compacted(len(kwargs.get("input", [])))


class FakeAsyncOpenAI:
    def __init__(self):
        self.responses = FakeAsyncResponses()


fake = FakeAsyncOpenAI()
store = SQLiteSession("case_4711", WORKDIR / "compacting.db")
compacting = OpenAIResponsesCompactionSession(
    session_id="case_4711",
    underlying_session=store,
    client=fake,                                   # live: omit (uses the default AsyncOpenAI client)
    model=MODEL,                                   # the default would be "gpt-4.1"
    compaction_mode="input",                       # send the stored items, not a previous_response_id
    should_trigger_compaction=lambda ctx: len(ctx["compaction_candidate_items"]) >= 6,
)
for i, q in enumerate(["return window", "torn tent", "refund timing"]):
    agent = Agent(name="Support", instructions=RULES,
                  model=ScriptedModel([("search_kb", {"query": q})], final=f"Answer {i + 1}."), tools=[search_kb])
    await Runner.run(agent, f"Question {i + 1}", session=compacting, run_config=OFFLINE)
    kinds = [it.get("type") or it.get("role") for it in await store.get_items()]
    print(f"after turn {i + 1}: compact calls={len(fake.responses.calls)} stored={kinds}")
print("compact() was called with:", {k: (len(v) if isinstance(v, list) else v) for k, v in fake.responses.calls[0].items()})

> **Observe:** after turn 2 the candidate items (everything except user messages and compaction items) reached 6, so the session sent the stored items to `compact()` and replaced the stored history with the compacted window. Turn 3 then appended normally after the compaction item. You can also call `await compacting.run_compaction({"force": True})` yourself, for example at the end of a workflow phase.

Two more things to know: the wrapper refuses an `OpenAIConversationsSession` (that history lives on OpenAI's servers), and `ModelSettings` can enable server-side compaction per agent without any wrapper.

In [ ]:
try:
    server_side = OpenAIConversationsSession(openai_client=FakeAsyncOpenAI())   # history lives on OpenAI's servers
    OpenAIResponsesCompactionSession("x", server_side, model=MODEL)
except ValueError as e:
    print("ValueError:", e)

print(ModelSettings(context_management=[{"type": "compaction", "compact_threshold": 150_000}]).context_management)

> **Try it:** change `recent_turns` to 2 in the first demo and predict how many outputs get trimmed before you run it. Then make `should_trigger_compaction` use an estimated token count of `ctx["session_items"]` instead of an item count.

## 11.6 The case-facts block

Transactional facts (IDs, amounts, dates, requests, commitments, status) live **outside** the history in a structured object your code owns. Update them from tool results (the source of truth) and from a small structured extraction, merge in code, version every change, and render the block into every request. [Theory](README.md#116-the-case-facts-block)

In [ ]:
class CaseFacts(BaseModel):
    customer_id: str
    order_id: str | None = None
    order_date: str | None = None          # ISO 8601, as stated by a tool, not by the user
    order_total: str | None = None         # keep money as an exact string, e.g. "74.95 USD"
    issue: str | None = None
    customer_request: str | None = None
    commitments: list[str] = Field(default_factory=list)   # what we promised, with dates
    status: str = "open"
    version: int = 1
    conflicts: list[str] = Field(default_factory=list)     # disagreements to escalate, never silently resolved


class FactsDelta(BaseModel):
    """What a structured extraction may propose from one user message. All fields nullable."""
    order_id: str | None = None
    order_total: str | None = None
    issue: str | None = None
    customer_request: str | None = None


TOOL_OWNED = {"order_id", "order_date", "order_total"}     # fields only a system of record may set
CHANGE_LOG: list[dict] = []


def render_facts(f: CaseFacts) -> str:
    lines = [f"{k}: {v}" for k, v in f.model_dump().items() if v not in (None, [], "") and k != "version"]
    return f'<case_facts version="{f.version}">\n' + "\n".join(lines) + "\n</case_facts>"


def merge_facts(facts: CaseFacts, delta: dict, source: str) -> CaseFacts:
    """source='tool' overwrites; source='user' may not overwrite a tool-owned value that differs."""
    new = facts.model_copy(deep=True)
    changed = False
    for key, value in delta.items():
        if value is None:
            continue
        if key == "commitments":
            for c in value:
                if c not in new.commitments:
                    new.commitments.append(c)
                    changed = True
            continue
        current = getattr(new, key)
        if current == value:
            continue
        if source == "user" and key in TOOL_OWNED and current is not None:
            new.conflicts.append(f"{key}: user says {value!r}, system of record says {current!r}")
            changed = True
            continue
        setattr(new, key, value)
        changed = True
    if changed:
        new.version += 1
        CHANGE_LOG.append({"version": new.version, "source": source, "delta": delta})
    return new


facts = CaseFacts(customer_id="CUST-881")
facts = merge_facts(facts, {"order_id": "A-1001", "issue": "tent arrived torn", "customer_request": "refund"}, "user")
order = full_order_record("A-1001")
facts = merge_facts(facts, {"order_total": order["total"], "order_date": order["placed_at"][:10]}, "tool")
facts = merge_facts(facts, {"order_total": "120 USD"}, "user")            # the customer misremembers
facts = merge_facts(facts, {"commitments": ["2026-10-03: refund of 74.95 USD requested, 5-7 business days"]}, "agent")
print(render_facts(facts))
print("\nchange log:", [(c["version"], c["source"]) for c in CHANGE_LOG])

> **Observe:** the user's "$120" did not overwrite the order system's 74.95 USD; it became a recorded **conflict** to show to a human. Every change bumped the version and was logged, which is your audit trail (the compaction item cannot be one).

**Where to put the block?** Compare how much of the request prefix stays identical between two consecutive turns, which is what prompt caching can reuse.

In [ ]:
def rendered_prefix(request: dict) -> list[str]:
    """Flatten a request into comparable chunks, in the order the model reads them."""
    return [request.get("instructions") or ""] + [json.dumps(i, sort_keys=True) for i in request["input"]]


def shared_prefix_tokens(a: dict, b: dict) -> int:
    n = 0
    for x, y in zip(rendered_prefix(a), rendered_prefix(b)):
        if x != y:
            break
        n += est_tokens(x)
    return n


def build_input(stored_history: list, f: CaseFacts, user_message: str) -> list:
    return [*stored_history,
            {"role": "developer", "content": render_facts(f)},   # ephemeral: NOT saved to history
            msg("user", user_message)]


f1 = facts
f2 = merge_facts(f1, {"customer_request": "refund to the original card"}, "user")
next_history = [*HISTORY, msg("user", "Can I still return it?"), msg("assistant", "Yes.")]

in_instructions = (
    {"instructions": RULES + "\n" + render_facts(f1), "input": [*HISTORY, msg("user", "Can I still return it?")]},
    {"instructions": RULES + "\n" + render_facts(f2), "input": [*next_history, msg("user", "Refund to my card.")]},
)
ephemeral_end = (
    {"instructions": RULES, "input": build_input(HISTORY, f1, "Can I still return it?")},
    {"instructions": RULES, "input": build_input(next_history, f2, "Refund to my card.")},
)
for name, (a, b) in {"facts in instructions": in_instructions, "ephemeral at the end": ephemeral_end}.items():
    print(f"{name:<22} shared prefix between turns: ~{shared_prefix_tokens(a, b):>5,} of ~{est_tokens(b['input']):,} est. tokens")

> **Observe:** facts at the top of `instructions` break the shared prefix at the very first token whenever they change; the ephemeral developer message at the end keeps the whole history reusable. That is why it is the default for stateless and Agents SDK designs.

**(live)** The extraction half of the update loop: a small structured call that proposes a delta. Your code, not the model, decides what gets merged.

In [ ]:
# (live) structured extraction of a FactsDelta from one user message
parsed = client.responses.parse(
    model=MODEL, store=False,
    reasoning={"effort": "low"},      # extraction is easy: choose the effort instead of inheriting the default (medium)
    max_output_tokens=2_000,          # room for reasoning + JSON; a tight cap can end in status="incomplete"
    instructions="Extract only facts the customer states explicitly. Use null for anything not stated.",
    input="Hi, it's about order A-1001. The zipper on the tent is broken, I'd like a replacement rather than a refund.",
    text_format=FactsDelta,
)
refusal = next((c.refusal for o in parsed.output if o.type == "message" for c in o.content if c.type == "refusal"), None)
if parsed.status != "completed":
    print("incomplete, nothing merged:", parsed.status, parsed.incomplete_details)   # never invent a delta
elif refusal:
    print("refused, nothing merged:", refusal)
elif parsed.output_parsed is None:
    print("no parsed output, nothing merged")
else:
    delta = parsed.output_parsed
    print(delta)
    print(render_facts(merge_facts(facts, delta.model_dump(), "user")))

> **Try it:** send a message that states a different order total and confirm it lands in `conflicts`, not in `order_total`.

## 11.7 Trimming verbose tool outputs

Trim at three levels: **at the source** (the tool returns only the fields the agent needs), **in older turns** (replace old outputs with a preview, never delete the item), and **into facts** (once a result has updated the case-facts block, the full output can go). Keep `function_call` / `function_call_output` pairs intact, never trim the current turn, and keep error outputs readable. [Theory](README.md#117-trimming-verbose-tool-outputs)

In [ ]:
ORDER_FIELDS = ("order_id", "status", "total", "placed_at", "refund_window_ends")


def lookup_order_raw(order_id: str) -> str:
    return json.dumps(full_order_record(order_id))


def lookup_order(order_id: str) -> str:
    """Look up an order by ID. Returns status, total, order date and refund deadline."""
    record = full_order_record(order_id)
    return json.dumps({k: record.get(k) for k in ORDER_FIELDS})


raw, slim = lookup_order_raw("A-1001"), lookup_order("A-1001")
print(f"raw: ~{est_tokens(raw)} est. tokens   at-source trim: ~{est_tokens(slim)} est. tokens")
print("over 30 turns that is ~", f"{30 * (est_tokens(raw) - est_tokens(slim)):,}", "fewer tokens in EVERY later request")
print(slim)

For a hand-written stateless loop, do what `ToolOutputTrimmer` does: replace the `output` string of **old** outputs, keep the items and their `call_id`s, and leave errors alone.

In [ ]:
def trim_old_outputs(items: list[dict], recent_turns: int = 1, max_chars: int = 400, preview_chars: int = 80) -> list[dict]:
    user_idx = [i for i, it in enumerate(items) if it.get("role") == "user"]
    boundary = user_idx[-recent_turns] if len(user_idx) >= recent_turns else 0
    names = {it["call_id"]: it["name"] for it in items if it.get("type") == "function_call"}
    out = []
    for i, it in enumerate(items):
        text = it.get("output")
        if (i < boundary and it.get("type") == "function_call_output" and isinstance(text, str)
                and len(text) > max_chars and '"error"' not in text):
            it = {**it, "output": f"[Trimmed: {names.get(it['call_id'], 'tool')} output — {len(text)} chars → "
                                  f"{preview_chars} char preview]\n{text[:preview_chars]}..."}
        out.append(it)
    return out


def check_pairs(items: list[dict]) -> list[str]:
    calls = {it["call_id"] for it in items if it.get("type") == "function_call"}
    outputs = {it["call_id"] for it in items if it.get("type") == "function_call_output"}
    return [f"orphan call {c}" for c in calls - outputs] + [f"orphan output {c}" for c in outputs - calls]


error_turn = [msg("user", "And order B-9?"),
              {"type": "function_call", "call_id": "call_err", "name": "lookup_order", "arguments": '{"order_id": "B-9"}'},
              {"type": "function_call_output", "call_id": "call_err",
               "output": json.dumps({"error": "order B-9 not found", "retryable": False, "detail": "x" * 500})},
              msg("assistant", "I could not find B-9.")]
history_with_error = [*HISTORY[:4], *error_turn, *HISTORY[4:]]
trimmed = trim_old_outputs(history_with_error, recent_turns=1)
print(f"before: ~{est_tokens(history_with_error):,}  after: ~{est_tokens(trimmed):,} est. tokens  "
      f"| items {len(history_with_error)} -> {len(trimmed)} | pair problems: {check_pairs(trimmed) or 'none'}")
print("error output kept readable:", trimmed[6]["output"][:60])
print("last turn untouched:", trimmed[-2]["output"] == history_with_error[-2]["output"])

> **Observe:** the history shrank, item count and `call_id` pairs are unchanged, the error output is still readable, and the current turn is intact. Trimming changes the prefix, so the next request misses the cache from the first changed item: trim in **batches** (for example, together with compaction), not one item per turn.

## 11.8 Position-aware input

Models use the beginning and the end of a long input more reliably than the middle. Put stable rules first (also the cacheable prefix), long material in the middle, and the dynamic critical parts (case facts, latest tool results, the question) at the end. For aggregated results, lead with key findings and restate the action items at the end. [Theory](README.md#118-position-aware-input)

In [ ]:
REPORTS = [
    {"file": "cache.py", "severity": "low", "finding": "cache TTL hard-coded to 24 hours", "detail": "..." * 200},
    {"file": "login.py", "severity": "critical", "finding": "session token written to the access log", "detail": "..." * 200},
    {"file": "inventory.py", "severity": "high", "finding": "stock reservation retried with no backoff", "detail": "..." * 200},
]
SEVERITY = {"critical": 0, "high": 1, "medium": 2, "low": 3}


def aggregate(reports: list[dict], question: str) -> str:
    ranked = sorted(reports, key=lambda r: SEVERITY[r["severity"]])
    key = "\n".join(f"{n}. {r['file']}: {r['finding']} ({r['severity']})" for n, r in enumerate(ranked[:3], 1))
    detail = "\n".join(f'<findings file="{r["file"]}">{r["detail"][:60]}...</findings>' for r in ranked)
    return (f"## SUMMARY (read first)\n{key}\n\n## EVIDENCE\n{detail}\n\n"
            f"## TASK (restated)\n{question} Answer only from the findings above; cite file names.")


print(aggregate(REPORTS, "Which issue must be fixed before the release?"))

A linter for the layout of a request that asks a **new question**. It flags the two mistakes from the README: dynamic data at the top of `instructions`, and a question stated only **before** the long context (OpenAI's long-context advice: instructions at both ends). It does not insist that a user message is the last item: a tool-loop continuation ends with a `function_call_output`, an explicit compaction ends with a `compaction_trigger`, and the 11.5 filter appends ephemeral facts after the question. All of those are valid.

In [ ]:
DYNAMIC_MARKERS = ("<case_facts", "Current time", "timestamp", "Customer name:")


def position_lint(request: dict) -> list[str]:
    problems = []
    instr = request.get("instructions") or ""
    if any(m in instr[:300] for m in DYNAMIC_MARKERS):
        problems.append("dynamic data at the top of instructions: every change invalidates the cached prefix")
    items = request.get("input", [])
    # long CONTEXT = big user/developer messages (documents); tool outputs in a tool loop are not "context placement"
    long_idx = [i for i, it in enumerate(items) if it.get("type", "message") == "message"
                and it.get("role") in ("user", "developer") and est_tokens(it) > 500]
    q_idx = [i for i, it in enumerate(items) if it.get("role") == "user" and est_tokens(it) <= 500]
    if long_idx and q_idx and max(q_idx) < max(long_idx):
        problems.append("the question appears only before the long context: restate it after the context too (both ends)")
    if items and items[-1].get("role") == "assistant":
        problems.append("the request ends with an assistant message: there is nothing new for the model to act on")
    return problems


docs = [msg("user", "Reference documents:\n" + "Policy text. " * 400)]
good = {"instructions": RULES, "input": [*docs, {"role": "developer", "content": render_facts(facts)},
                                         msg("user", "Can I get a refund?")]}
bad = {"instructions": render_facts(facts) + "\n" + RULES, "input": [msg("user", "Can I get a refund?"), *docs]}
continuation = {"instructions": RULES, "input": [*good["input"],
    {"type": "function_call", "call_id": "call_r", "name": "lookup_order", "arguments": '{"order_id": "A-1001"}'},
    {"type": "function_call_output", "call_id": "call_r", "output": lookup_order("A-1001")}]}
facts_after_question = {"instructions": RULES, "input": [*docs, msg("user", "Can I get a refund?"),
                                                         {"role": "developer", "content": render_facts(facts)}]}
print("good:                ", position_lint(good) or "OK")
print("bad:                 ", position_lint(bad))
print("tool continuation:   ", position_lint(continuation) or "OK")
print("facts after question:", position_lint(facts_after_question) or "OK")

> **Observe:** the caching rule (stable first, dynamic last) and the attention rule (critical at the start or end) agree. The placement advice comes from OpenAI's GPT-4.1 prompting guide, an older model; validate it on GPT-6 with your own evals.

## 11.9 Scratchpad files and memory across sessions

A scratchpad is storage you own where the agent records verified findings with sources, so a new session (or a session after compaction) reads them instead of re-discovering them. When external state has moved on, start a **fresh** session seeded with a structured summary instead of resuming stale tool results. [Theory](README.md#119-scratchpad-files-and-memory-across-sessions)

In [ ]:
NOTES = WORKDIR / "case-notes" / "findings.md"


@function_tool
def append_note(finding: str, source: str) -> str:
    """Record one verified finding with its source (file:line, ticket, URL). Use for facts you will need later."""
    NOTES.parent.mkdir(parents=True, exist_ok=True)
    with NOTES.open("a") as f:
        f.write(f"- {finding} (source: {source})\n")
    return "saved"


@function_tool
def read_notes() -> str:
    """Read all findings recorded so far in this investigation. Call this before re-running discovery."""
    return NOTES.read_text() if NOTES.exists() else "(no notes yet)"


# Session 1: the agent records two findings.
plan = [("append_note", {"finding": "stock reservation is retried with no backoff", "source": "inventory.py:88"}),
        ("append_note", {"finding": "login.py writes the session token to the access log", "source": "login.py:41"})]
s1 = Agent(name="Investigator", instructions="Record findings as you go.", model=ScriptedModel(plan, final="Logged."),
           tools=[append_note, read_notes])
await Runner.run(s1, "Investigate the checkout slowdown.", run_config=OFFLINE)

# Session 2: a brand-new session (no shared history) starts by reading the notes.
model2 = ScriptedModel([("read_notes", {})], final="Resuming from 2 recorded findings.")
s2 = Agent(name="Investigator", instructions="Read your notes before doing anything else.", model=model2,
           tools=[append_note, read_notes])
r2 = await Runner.run(s2, "Continue the investigation.", run_config=OFFLINE)
print(NOTES.read_text())
print("session 2 saw the notes:", "backoff" in str(model2.calls[-1]["input"]), "|", r2.final_output)

**Resume or start fresh?** A small decision helper: resume only when nothing the session relied on has changed; otherwise seed a new session with facts, notes and "what changed".

In [ ]:
def resume_or_fresh(checkpoint: dict, external: dict) -> tuple[str, str | None]:
    changed = [k for k, v in checkpoint["observed"].items() if external.get(k) != v]
    if not changed:
        return "resume", None
    seed = (f"{render_facts(facts)}\n\n## Notes\n{read_notes_text()}\n\n"
            f"## What changed since {checkpoint['at']}\n" + "\n".join(f"- {k}: {checkpoint['observed'][k]} -> {external.get(k)}" for k in changed))
    return "fresh", seed


def read_notes_text() -> str:
    return NOTES.read_text() if NOTES.exists() else "(no notes yet)"


cp = {"at": "2026-10-02T17:00Z", "observed": {"order A-1001 status": "delivered", "refund-4711-1": "requested"}}
print(resume_or_fresh(cp, {"order A-1001 status": "delivered", "refund-4711-1": "requested"})[0])
decision, seed = resume_or_fresh(cp, {"order A-1001 status": "delivered", "refund-4711-1": "completed"})
print(decision, "\n" + seed)

> **Observe:** Codex has the same vocabulary for coding sessions: `/compact`, `/status` (remaining context), `/new`, `/resume`, `/fork`, and `AGENTS.md` for durable project guidance (chapter 05).

## 11.10 Delegating to sub-agents to protect context

Verbose exploration belongs in a sub-agent with its own context and a minimal toolset. With an **agent-as-tool**, only the tool result returns to the coordinator, and `custom_output_extractor` controls exactly what that result is. Delegation adds total tokens; it protects the coordinator's context. [Theory](README.md#1110-delegating-to-sub-agents-to-protect-context)

In [ ]:
FILES = {
    "inventory/service.py": "import warehouse\nimport stock\n" + "# business logic\n" * 150,
    "inventory/warehouse.py": "import httpx\nimport retry_policy\n" + "# HTTP client code\n" * 150,
    "inventory/stock.py": "import db\n" + "# stock counts per SKU\n" * 150,
}


@function_tool
def read_file(path: str) -> str:
    """Read one file from the repository."""
    return FILES.get(path, f"(no such file: {path})")


class DependencyReport(BaseModel):
    module: str
    depends_on: list[str]
    evidence: list[str]          # file:line references
    gaps: list[str]              # what could not be checked


report = DependencyReport(module="inventory", depends_on=["warehouse", "stock", "httpx", "db"],
                          evidence=["inventory/service.py:1-2", "inventory/warehouse.py:1", "inventory/stock.py:1"],
                          gaps=["retry_policy not inspected"])
explorer_model = ScriptedModel([("read_file", {"path": p}) for p in FILES], final=report.model_dump_json())
explorer = Agent(name="Dependency explorer", model=explorer_model, output_type=DependencyReport, tools=[read_file],
                 instructions="Map what the given module imports. Return only the structured report.")


async def compact_report(result) -> str:
    """One compact JSON line for the coordinator: keep the evidence (file:line) so findings stay verifiable."""
    r = result.final_output
    return json.dumps({"module": r.module, "depends_on": r.depends_on,
                       "evidence": r.evidence[:10], "gaps": r.gaps}, separators=(",", ":"))


coordinator_model = ScriptedModel([("explore_dependencies", {"input": "inventory"})], final="Plan drafted.")
coordinator = Agent(
    name="Coordinator", model=coordinator_model,
    instructions="Plan the change. Delegate exploration; never read more than 3 files yourself.",
    tools=[explorer.as_tool(tool_name="explore_dependencies",
                            tool_description="Investigate a module's dependencies and return a short report.",
                            custom_output_extractor=compact_report)],
)
result = await Runner.run(coordinator, "We need to change how stock reservations are retried.", run_config=OFFLINE)
returned = coordinator_model.calls[-1]["input"][-1]["output"]
print("coordinator received:", returned)
print(f"explorer context on its last call: ~{est_tokens(explorer_model.calls[-1]['input']):,} est. tokens")
print(f"coordinator context on its last call: ~{est_tokens(coordinator_model.calls[-1]['input']):,} est. tokens")

> **Observe:** the explorer read every file in its own context; the coordinator received one compact structured line with the dependencies, the file:line evidence and the gaps. That is the whole point: the coordinator's later reasoning stays specific instead of drifting into "typical patterns".

Two other mechanisms, shown as payloads only:

- A **handoff** passes the conversation history to the specialist; filter it with `input_filter` (for example `agents.extensions.handoff_filters.remove_all_tools`).
- **Responses Multi-agent** (beta) lets the model spawn subagents, each with its own context and its own automatic compaction. `/responses/compact` is not supported in this mode. Available on `gpt-6.1-sol` and the GPT-5.6 models, not on Luna.

In [ ]:
from agents import handoff
from agents.extensions.handoff_filters import remove_all_tools

refunds_specialist = Agent(name="Refunds", instructions="Handle refunds.", model=ScriptedModel(final="ok"))
h = handoff(refunds_specialist, input_filter=remove_all_tools)   # the specialist sees no earlier tool items
print("handoff tool:", h.tool_name, "| input_filter:", h.input_filter.__name__)

multi_agent_request = dict(   # client.beta.responses.create(**multi_agent_request) -- not sent here
    model="gpt-6.1-sol",
    input="Audit the inventory, login and cache modules in parallel; reconcile findings.",
    multi_agent={"enabled": True, "max_concurrent_subagents": 3},
    betas=["responses_multi_agent=v1"],
)
print(json.dumps(multi_agent_request, indent=2))

> **Try it:** give `DependencyReport` a long `raw_notes: str` field, return `result.final_output.model_dump_json()` from `compact_report`, and compare the coordinator's context size. Decide which fields the coordinator really needs, but keep the evidence: a result without sources cannot be checked.

## 11.11 Structured state persistence and crash recovery

Persist **your** state at well-defined checkpoints: a manifest of phases, the versioned case facts, per-worker results, and the serialized `RunState` of a paused run. Write atomically, give every side effect an idempotency key, and on restore rebuild the request instead of replaying everything. [Theory](README.md#1111-structured-state-persistence-and-crash-recovery)

In [ ]:
STATE = WORKDIR / "checkpoints"


def atomic_write_json(path: Path, data: dict) -> None:
    """Write to a temp file, then rename: a crash never leaves a half-written checkpoint."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2))
    os.replace(tmp, path)


class PaymentGateway:
    """Fake payment system that honors idempotency keys, like real payment APIs do."""

    def __init__(self):
        self.refunds: dict[str, dict] = {}
        self.calls = 0

    def refund(self, key: str, order_id: str, amount: str) -> dict:
        self.calls += 1
        return self.refunds.setdefault(key, {"order_id": order_id, "amount": amount, "status": "completed"})

    def status(self, key: str) -> str | None:
        return self.refunds.get(key, {}).get("status")


manifest = {
    "case_id": "case-4711", "updated_at": "2026-10-03T14:05:00Z",
    "phases": {"triage": "done", "refund_check": "running", "customer_reply": "pending"},
    "owners": {"triage": "coordinator", "refund_check": "refund_specialist"},   # who owns each phase (11.10)
    "facts_version": facts.version,
    "side_effects": [{"id": "refund-4711-1", "status": "requested", "order_id": "A-1001", "amount": "74.95 USD"}],
}
atomic_write_json(STATE / "manifest.json", manifest)
atomic_write_json(STATE / "case-4711.facts.json", facts.model_dump())
print(sorted(p.name for p in STATE.iterdir()))

Simulate a crash: the refund request reached the gateway, but the process died before the manifest was updated. On restore, check the external system before repeating anything.

In [ ]:
gateway = PaymentGateway()
gateway.refund("refund-4711-1", "A-1001", "74.95 USD")      # happened just before the crash


def restore(state_dir: Path, gw: PaymentGateway) -> dict:
    m = json.loads((state_dir / "manifest.json").read_text())
    f = CaseFacts.model_validate_json((state_dir / "case-4711.facts.json").read_text())
    for se in m["side_effects"]:
        if se["status"] == "requested":
            if gw.status(se["id"]) == "completed":
                se["status"] = "completed"                      # verified, do NOT repeat
            else:
                gw.refund(se["id"], se["order_id"], se["amount"])  # same key: safe even if it did go through
                se["status"] = "completed"
    todo = [p for p, s in m["phases"].items() if s != "done"]
    owners = {p: m.get("owners", {}).get(p, "coordinator") for p in todo}   # who resumes, with which context
    atomic_write_json(state_dir / "manifest.json", m)
    return {"facts_version": f.version, "resume_phases": todo, "resume_owners": owners,
            "side_effects": m["side_effects"]}


print(restore(STATE, gateway))
print("refund calls made to the gateway:", gateway.calls, "| refunds issued:", len(gateway.refunds))

A run paused for human approval survives a restart through `RunState`: `result.to_state().to_string()` to save, `await RunState.from_string(agent, saved)` to load (in a different process; `from_string` and `from_json` are coroutines in `openai-agents` 0.23), then approve or reject and run from that state.

In [ ]:
from agents import RunState


def build_refund_agent(model: Model) -> Agent:
    @function_tool(needs_approval=True)
    def issue_refund(order_id: str, amount: str) -> str:
        """Issue a refund. Requires manager approval."""
        return json.dumps(gateway.refund(f"refund-{order_id}-1", order_id, amount))

    return Agent(name="Refunds", instructions="Refund only after approval.", model=model, tools=[issue_refund])


# Process 1: the run pauses for approval; we persist the state and exit.
agent_p1 = build_refund_agent(ScriptedModel([("issue_refund", {"order_id": "A-17", "amount": "42.00 USD"})]))
paused = await Runner.run(agent_p1, "Refund order A-17", run_config=OFFLINE)
print("interruptions:", [i.name for i in paused.interruptions])
(STATE / "run-4711.state").write_text(paused.to_state().to_string())

# Process 2 (after a redeploy): rebuild the agent, load the state, approve, continue.
agent_p2 = build_refund_agent(ScriptedModel(final="Refund for A-17 issued."))
state = await RunState.from_string(agent_p2, (STATE / "run-4711.state").read_text())   # async in openai-agents 0.23
for item in state.get_interruptions():
    print("approving:", item.name, item.arguments)
    state.approve(item)                                         # or state.reject(item)
resumed = await Runner.run(agent_p2, state, run_config=OFFLINE)
print("final:", resumed.final_output, "| gateway refunds:", list(gateway.refunds))

> **Observe:** process 2 never saw the original conversation in memory; it continued from the serialized state. A `previous_response_id` alone could not do this (it is not the SDK's run state, and it does not stop a double refund).

## 11.12 Context management and prompt caching

Every context-management action edits the prompt, and caching rewards prompts that do not change. Keep a stable prefix, keep history append-only between compactions, keep tools stable (`tool_choice` / `allowed_tools` instead of removing definitions), and change effort with a `configuration_update` item. Then quantify: compaction resets part of the cache, but much smaller requests are usually cheaper overall. [Theory](README.md#1112-context-management-and-prompt-caching)

In [ ]:
def input_cost(uncached: int, cached: int, writes: int, rate_per_m: float, read_mult: float = 0.1) -> float:
    """GPT-5.6+ pricing: reads 0.1x (0.05x on gpt-6.1-sol), writes 1.25x."""
    return (uncached + read_mult * cached + 1.25 * writes) * rate_per_m / 1e6


# Implicit caching (the default on GPT-5.6+): each turn's new suffix, up to the latest user message,
# is written to the cache at 1.25x; the reused prefix is read at 0.1x.
before = input_cost(uncached=0, cached=177_000, writes=3_000, rate_per_m=10.0)       # gpt-6-astra, 180K input
after_first = input_cost(uncached=0, cached=0, writes=30_000, rate_per_m=10.0)
after_next = input_cost(uncached=0, cached=27_000, writes=3_000, rate_per_m=10.0)   # same 3K new tokens per turn
print(f"{before:.3f} vs {after_first:.3f} then {after_next:.3f} per turn")

Over a run of turns, compare total cost with and without one compaction (Astra prices, implicit caching: 3,000 new tokens per turn written to the cache, everything else read from it). The compaction pass is a **billed request** too, so the simulation charges it: its input is the whole history, and its output tokens are an assumption here. Measure the real numbers from `compacted.usage` (standalone) or from the usage of the response that compacted (server-side).

In [ ]:
ASTRA_OUTPUT_RATE = 50.0   # USD per 1M output tokens


def simulate(turns: int, start: int, compact_at: int | None, compacted_size: int = 30_000, new: int = 3_000,
             compaction_input_cached: bool = True, compaction_output: int = 5_000) -> float:
    size, total = start, 0.0
    for t in range(1, turns + 1):
        if compact_at is not None and t == compact_at:
            # 1) the compaction pass itself: reads the whole history, generates the compaction item
            reads = input_cost(0, size, 0, 10.0) if compaction_input_cached else input_cost(size, 0, 0, 10.0)
            total += reads + compaction_output * ASTRA_OUTPUT_RATE / 1e6
            # 2) the first turn after compaction rewrites the cache
            size = compacted_size
            total += input_cost(0, 0, size + new, 10.0)
        else:
            total += input_cost(0, size, new, 10.0)              # read the prefix, write the new suffix
        size += new
    return total


def payback_turns(**kw) -> int | None:
    return next((n for n in range(1, 201) if simulate(n, 180_000, 1, **kw) < simulate(n, 180_000, None)), None)


print("assumed compaction pass: input = the 180K history, 5,000 output tokens (measure yours)")
for label, cached in (("compaction input read from cache", True), ("compaction input uncached (worst case)", False)):
    a, b = simulate(20, 180_000, None), simulate(20, 180_000, 1, compaction_input_cached=cached)
    print(f"  {label:<40} 20 turns: ${a:.2f} without vs ${b:.2f} with; pays back after "
          f"{payback_turns(compaction_input_cached=cached)} turns")

> **Observe:** the compaction pass and the turn right after it cost more, every later turn costs much less, and the context is smaller (better quality). Whether it pays back depends on the cost of the compaction pass and on how many turns are left: a handful of turns if the pass reads the history from cache, many more in the worst case, and never for a conversation that is about to end. Measure your own workload with the compaction usage, `cached_tokens` and `cache_write_tokens` before you quote a payback period.

What breaks the cache between two consecutive requests? A linter that mirrors the caching guide:

In [ ]:
CACHE_KEYS = ("model", "tools", "parallel_tool_calls", "text", "context_management", "instructions")


def cache_breakers(prev: dict, nxt: dict) -> list[str]:
    reasons = [f"{k} changed" for k in CACHE_KEYS if prev.get(k) != nxt.get(k)]
    if (prev.get("reasoning") or {}).get("effort") != (nxt.get("reasoning") or {}).get("effort"):
        reasons.append("reasoning.effort changed: append a configuration_update item instead")
    p_items, n_items = prev.get("input", []), nxt.get("input", [])
    if n_items[:len(p_items)] != p_items:
        reasons.append("history was rewritten, not appended (expected after compaction/trimming: do it in batches)")
    return reasons


base_req = {"model": MODEL, "instructions": RULES, "tools": TOOLS, "reasoning": {"effort": "low"}, "input": HISTORY}
variants = {
    "append-only next turn": {**base_req, "input": [*HISTORY, msg("user", "Next?")]},
    "effort raised in request": {**base_req, "reasoning": {"effort": "high"}, "input": [*HISTORY, msg("user", "Next?")]},
    "effort via configuration_update": {**base_req, "input": [*HISTORY, {"type": "configuration_update",
                                                                        "reasoning": {"effort": "high"}}, msg("user", "Next?")]},
    "a tool removed": {**base_req, "tools": TOOLS[:1], "input": [*HISTORY, msg("user", "Next?")]},
    "old outputs trimmed": {**base_req, "input": [*trim_old_outputs(HISTORY, 1), msg("user", "Next?")]},
}
for name, r in variants.items():
    print(f"{name:<32} {cache_breakers(base_req, r) or 'cache-friendly'}")

> **Gotcha:** a timestamp, a user name or the case-facts block at the top of `instructions` turns every request into a cache miss (11.6, 11.8). Remember too that cached tokens still count toward tokens-per-minute rate limits.

## Mini-project: a long-running support conversation manager

Everything above in one small class. It:

1. keeps a versioned **case-facts block** outside the history, updated from tool results (11.6);
2. trims `lookup_order` **at the source** and stores slim outputs (11.7);
3. **measures** every request against a budget, **compacts** at the soft limit, and **refuses to send** anything still above the hard limit after compacting (11.1, 11.4);
4. sends facts as an **ephemeral** developer message at the end, so the history prefix stays stable (11.6, 11.12);
5. **checkpoints atomically** after every turn and **restores** after a simulated crash (11.11).

It runs offline with token estimates, a stub model and a stub compactor. The backend is pluggable; the last cell swaps in the live one. The toy budget is tiny (soft limit 700 estimated tokens) so that compaction triggers within a few turns.

In [ ]:
class OfflineBackend:
    """Token estimates, a stub compactor and a stub model. Same interface as LiveBackend below."""

    def count(self, instructions: str, items: list[dict]) -> int:
        return est_tokens(instructions) + est_tokens(items)

    def compact(self, items: list[dict]) -> list[dict]:
        last_user = [i for i in items if i.get("role") == "user"][-1:]
        digest = hashlib.sha256(json.dumps(items).encode()).hexdigest()
        return [*last_user, {"type": "compaction", "encrypted_content": f"opaque-{digest[:24]}"}]

    def respond(self, instructions: str, items: list[dict]) -> list[dict]:
        facts_msg = next((i["content"] for i in reversed(items) if i.get("role") == "developer"), "")
        total = re.search(r"order_total: (.+)", facts_msg)
        question = items[-1]["content"]
        text = f"(stub) On '{question[:30]}': order total on file is {total.group(1) if total else 'unknown'}."
        return [assistant_item(text)]


@dataclass
class Budget:
    soft: int
    hard: int


class ContextBudgetExceeded(Exception):
    """The request is still above the hard limit after compaction: do not send it."""


def display_text(output: list[dict]) -> str:
    """Text to show the user. The full output items (not this string) are what gets stored and replayed."""
    for item in reversed(output):
        if item.get("type") == "message":
            parts = item.get("content", [])
            text = "".join(p.get("text") or "" for p in parts if p.get("type") == "output_text")
            refusal = " ".join(p.get("refusal") or "" for p in parts if p.get("type") == "refusal")
            return text or (f"[refusal] {refusal}" if refusal else "")
    return ""


class SupportConversationManager:
    def __init__(self, case_id: str, state_dir: Path, backend, budget: Budget,
                 facts: CaseFacts | None = None, history: list[dict] | None = None, turns: int = 0,
                 compactions: int = 0):
        self.case_id, self.state_dir, self.backend, self.budget = case_id, state_dir, backend, budget
        self.facts = facts or CaseFacts(customer_id="CUST-881")
        self.history = history or []          # append-only between compactions; facts are NOT stored here
        self.turns, self.compactions = turns, compactions

    # --- one turn --------------------------------------------------------------------------
    def turn(self, user_message: str, order_id: str | None = None) -> str:
        context_items = []
        if order_id:                                                       # app-side retrieval this turn
            slim = lookup_order(order_id)                                  # trimmed at the source
            record = json.loads(slim)
            self.facts = merge_facts(self.facts, {"order_id": record["order_id"],
                                                  "order_total": record["total"]}, "tool")
            context_items.append({"role": "developer", "content": f"<lookup_order>{slim}</lookup_order>"})
        user = msg("user", user_message)
        request_items = self._assemble(context_items, user)
        tokens = self.backend.count(RULES, request_items)
        action = "send" if tokens < self.budget.soft else ("compact" if tokens < self.budget.hard else "compact_now")
        if action != "send":
            if self.history:
                self.history = self.backend.compact(self.history)           # compact BEFORE we run out of room
                self.compactions += 1
            request_items = self._assemble(context_items, user)
            tokens = self.backend.count(RULES, request_items)
            if tokens >= self.budget.hard:
                # compacting old history cannot shrink THIS turn's input: never send it as-is
                raise ContextBudgetExceeded(
                    f"~{tokens:,} tokens after compaction (hard limit {self.budget.hard:,}); shorten or split the "
                    "new input, or store it and pass a reference")
        output = self.backend.respond(RULES, request_items)                 # every output item, reasoning included
        self.history += [*context_items, user, *output]                    # the facts message is NOT stored
        self.turns += 1
        self._checkpoint(tokens, action)
        return display_text(output)

    def _assemble(self, context_items: list[dict], user: dict) -> list[dict]:
        facts_msg = {"role": "developer", "content": render_facts(self.facts)}
        return [*self.history, *context_items, facts_msg, user]           # stable prefix ... dynamic tail

    # --- persistence -----------------------------------------------------------------------
    def _checkpoint(self, tokens: int, action: str) -> None:
        atomic_write_json(self.state_dir / f"{self.case_id}.json", {
            "case_id": self.case_id, "turns": self.turns, "compactions": self.compactions,
            "last_request_tokens": tokens, "last_action": action,
            "facts": self.facts.model_dump(), "history": self.history,
        })

    @classmethod
    def restore(cls, case_id: str, state_dir: Path, backend, budget: Budget) -> "SupportConversationManager":
        cp = json.loads((state_dir / f"{case_id}.json").read_text())
        return cls(case_id, state_dir, backend, budget, facts=CaseFacts.model_validate(cp["facts"]),
                   history=cp["history"], turns=cp["turns"], compactions=cp["compactions"])


print("manager defined")

> **Note:** to keep the mini-project small, the order lookup is **app-side retrieval** (your code fetches the slim record and adds it as a developer message) rather than a model-initiated tool call. In an agent loop the model would request `lookup_order`, and you would trim old `function_call_output` items as in 11.7; the facts, budget and persistence logic stay the same.

Run 12 turns, crash, restore, and keep going.

In [ ]:
MP_DIR = WORKDIR / "mini-project"
budget = Budget(soft=700, hard=1_500)
mgr = SupportConversationManager("case-4711", MP_DIR, OfflineBackend(), budget)

script = [("Hi, my tent from order A-1001 arrived torn.", "A-1001")] + [
    (f"Follow-up question {n} about the return label and pickup times.", "A-1001" if n % 3 == 0 else None)
    for n in range(1, 12)
]
for i, (text, oid) in enumerate(script[:8], 1):
    answer = mgr.turn(text, oid)
    cp = json.loads((MP_DIR / "case-4711.json").read_text())
    print(f"turn {i:>2}: ~{cp['last_request_tokens']:>5} est. tokens  action={cp['last_action']:<11} "
          f"history={len(mgr.history):>2} items  facts v{mgr.facts.version}")

print("\n*** simulated crash: the process dies, the manager object is gone ***\n")
del mgr

mgr = SupportConversationManager.restore("case-4711", MP_DIR, OfflineBackend(), budget)
for i, (text, oid) in enumerate(script[8:], 9):
    answer = mgr.turn(text, oid)
    cp = json.loads((MP_DIR / "case-4711.json").read_text())
    print(f"turn {i:>2}: ~{cp['last_request_tokens']:>5} est. tokens  action={cp['last_action']:<11} "
          f"history={len(mgr.history):>2} items  facts v{mgr.facts.version}")
print("\nlast answer:", answer)
print("compactions:", mgr.compactions, "| exact total survived:", mgr.facts.order_total == "74.95 USD")

turns_before = mgr.turns
try:
    mgr.turn("Here is my full chat export: " + "x" * 10_000)       # far above the hard limit on its own
except ContextBudgetExceeded as e:
    print("rejected before inference:", e, "| turns unchanged:", mgr.turns == turns_before)
print("facts ever stored inside the history?", any("case_facts" in json.dumps(i) for i in mgr.history))

> **Observe:** compaction kicked in at the soft limit and the request size dropped; the restored manager continued from the checkpoint with the same facts, history and counters; the exact amount came from the facts block every turn, not from compacted history; and a single oversized message was rejected before inference, because compacting old history cannot make a too-large new input fit.

**(live)** The same manager with a live backend: exact counts with `input_tokens.count`, real `responses.compact`, stateless `responses.create` with `store=False`. A real budget would use the README constants; this one stays small so that a few turns trigger one compaction (about $0.01 on `gpt-6-luna`).

In [ ]:
# (live) swap the backend; everything else is unchanged
class IncompleteResponse(Exception):
    """The model did not finish (for example status="incomplete", reason max_output_tokens)."""


class LiveBackend:
    def __init__(self, client: OpenAI, model: str):
        self.client, self.model = client, model

    def count(self, instructions: str, items: list[dict]) -> int:
        return self.client.responses.input_tokens.count(model=self.model, instructions=instructions, input=items).input_tokens

    def compact(self, items: list[dict]) -> list[dict]:
        if not items:
            return items
        compacted = self.client.responses.compact(model=self.model, input=items)
        return [o.model_dump(exclude_none=True) for o in compacted.output]    # canonical window: do not prune

    def respond(self, instructions: str, items: list[dict]) -> list[dict]:
        resp = self.client.responses.create(
            model=self.model, instructions=instructions, input=items, store=False,
            reasoning={"effort": "low"},      # a deliberate choice: Luna's default effort is medium
            max_output_tokens=2_000,          # room for reasoning + answer; a tight cap can end in status="incomplete"
        )
        if resp.status != "completed":        # never store a half-finished or invented turn
            raise IncompleteResponse(f"status={resp.status}, details={resp.incomplete_details}")
        # stateless replay: keep EVERY output item (encrypted reasoning items too); display_text() picks the text
        return [o.model_dump(exclude_none=True) for o in resp.output]


live = SupportConversationManager("case-live", WORKDIR / "mini-project-live", LiveBackend(client, MODEL),
                                  Budget(soft=1_200, hard=4_000))
for text, oid in script[:4]:
    print("->", live.turn(text, oid)[:120])
print("compactions:", live.compactions, "| facts:", live.facts.order_total,
      "| stored item types:", sorted({i.get("type", "message") for i in live.history}))

> **Try it:** (1) add a `commitments` update when the assistant promises a refund date; (2) trim old `function_call_output` items with `trim_old_outputs` at the same moment you compact (batching keeps the cache-friendly stretches long); (3) replace the hand-written loop with an Agents SDK agent using `OpenAIResponsesCompactionSession` and the `facts_and_trim` filter from 11.5.

## Recap

- A 1.05M window is not a plan: every turn re-bills the history, quality degrades with irrelevant context, and above 272K input tokens the whole request is billed at a higher rate. Measure with `input_tokens.count` and `usage`, and enforce a soft and a hard limit.
- Choose the state strategy by **who can edit the history**: stateless and Agents SDK sessions yes, Conversations partly, `previous_response_id` chains no. Do not mix strategies.
- Keep `truncation="disabled"`; `"auto"` silently drops the earliest, often most important, items.
- Server-side compaction: append output; in stateless mode you may drop items before the latest compaction item; never prune a chain or the output of `/responses/compact`. The compaction item is opaque.
- Agents SDK: `SessionSettings(limit)`, `session_input_callback` (once per run), `call_model_input_filter` (every model call), `ToolOutputTrimmer`, `OpenAIResponsesCompactionSession` (set its model and trigger).
- Facts that must survive exactly live in a versioned case-facts block updated from tools and rendered as an ephemeral message near the end.
- Trim tool output at the source and in older turns, keeping call/output pairs and errors intact; delegate verbose exploration to agents-as-tools with structured results.
- Checkpoint atomically, use idempotency keys, persist paused runs with `RunState`, and design context management and caching together.

Next: [homework](02_homework.ipynb)